# CNN 構成比較実験

MNIST データセットを使って、CNN の構成の違いが学習にどう影響するかを実験します。

## 実験一覧
1. **フィルタ数**: 少ない (8→16) vs 中 (16→32) vs 多い (64→128)
2. **フィルタサイズ**: 3×3 vs 5×5 vs 7×7
3. **層の深さ**: 2層 vs 3層 vs 4層
4. **フィルタ数の増やし方**: 層が深くなるほど増やす vs 一定 vs 減らす

## セットアップ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# デバイス設定
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"使用デバイス: {device}")

In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"訓練データ: {len(train_dataset)} 枚 / テストデータ: {len(test_dataset)} 枚")

## 共通の学習・評価・可視化関数

In [ ]:
def train_model(model, train_loader, epochs=EPOCHS, lr=LEARNING_RATE):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"  Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}")

    return losses


def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def plot_comparison(results, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name, data in results.items():
        ax1.plot(range(1, EPOCHS + 1), data["losses"], marker="o", label=name)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"{title} — 学習曲線")
    ax1.legend()
    ax1.grid(True)

    names = list(results.keys())
    accs = [results[n]["accuracy"] for n in names]
    params = [results[n]["params"] for n in names]
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"][:len(names)]
    bars = ax2.bar(names, accs, color=colors)
    ax2.set_ylabel("正解率 (%)")
    ax2.set_title(f"{title} — テスト正解率")
    ax2.set_ylim(max(0, min(accs) - 3), 100)
    for bar, acc, p in zip(bars, accs, params):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 f"{acc:.1f}%\n({p:,} params)", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()


def run_experiment(models, title):
    print(title)
    results = {}
    for name, model in models.items():
        print(f"\n--- {name} ---")
        p = count_params(model)
        print(f"  パラメータ数: {p:,}")
        losses = train_model(model, train_loader)
        acc = evaluate_model(model, test_loader)
        print(f"  テスト正解率: {acc:.2f}%")
        results[name] = {"losses": losses, "accuracy": acc, "params": p}
    return results


print("共通関数を定義しました。")

---
## 実験1: フィルタ数の比較

フィルタサイズ (3×3) と層数 (2層) は固定し、フィルタの枚数だけを変えて比較する。

| モデル | 構成 | 仮説 |
|--------|------|------|
| 少ない | Conv(8) → Conv(16) | MNIST は単純なので十分かも |
| 中 | Conv(16) → Conv(32) | 標準的な構成 |
| 多い | Conv(64) → Conv(128) | 過剰かも。パラメータが増えて遅くなる |

In [ ]:
class CNN_Filters(nn.Module):
    def __init__(self, f1, f2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, f1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(f1, f2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(f2 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


filter_models = {
    "少ない (8→16)": CNN_Filters(8, 16),
    "中 (16→32)": CNN_Filters(16, 32),
    "多い (64→128)": CNN_Filters(64, 128),
}

filter_results = run_experiment(filter_models, "実験1: フィルタ数の比較")

In [ ]:
plot_comparison(filter_results, "実験1: フィルタ数")

---
## 実験2: フィルタサイズの比較

フィルタの枚数 (16→32) は固定し、カーネルサイズを変えて比較する。

| モデル | カーネル | 1回で見る範囲 | 特徴 |
|--------|---------|-------------|------|
| 3×3 | 9ピクセル | 細かい特徴を検出 | パラメータ少ない |
| 5×5 | 25ピクセル | やや広い範囲 | |
| 7×7 | 49ピクセル | 広い範囲を一度に見る | パラメータ多い |

padding はサイズを維持するよう `kernel_size // 2` に設定。

In [ ]:
class CNN_KernelSize(nn.Module):
    def __init__(self, k):
        super().__init__()
        pad = k // 2  # サイズ維持のための padding
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, k, padding=pad), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, k, padding=pad), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


kernel_models = {
    "3×3": CNN_KernelSize(3),
    "5×5": CNN_KernelSize(5),
    "7×7": CNN_KernelSize(7),
}

kernel_results = run_experiment(kernel_models, "実験2: フィルタサイズの比較")

In [ ]:
plot_comparison(kernel_results, "実験2: フィルタサイズ")

---
## 実験3: 層の深さの比較

フィルタサイズ (3×3) は固定し、畳み込み層の数を変えて比較する。
各層の後に MaxPool すると画像が小さくなりすぎるので、Pool は2回までに制限。

| モデル | 構成 |
|--------|------|
| 2層 | Conv(16) → Pool → Conv(32) → Pool → FC |
| 3層 | Conv(16) → Pool → Conv(32) → Conv(64) → Pool → FC |
| 4層 | Conv(16) → Conv(32) → Pool → Conv(64) → Conv(128) → Pool → FC |

In [ ]:
class CNN_2Layer(nn.Module):
    """Conv→Pool→Conv→Pool"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 28→14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 14→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class CNN_3Layer(nn.Module):
    """Conv→Pool→Conv→Conv→Pool"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 28→14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 14→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class CNN_4Layer(nn.Module):
    """Conv→Conv→Pool→Conv→Conv→Pool"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 28→14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 14→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


depth_models = {
    "2層": CNN_2Layer(),
    "3層": CNN_3Layer(),
    "4層": CNN_4Layer(),
}

depth_results = run_experiment(depth_models, "実験3: 層の深さの比較")

In [ ]:
plot_comparison(depth_results, "実験3: 層の深さ")

---
## 実験4: フィルタ数の増やし方

3層構成で固定し、層が深くなるにつれてフィルタ数をどう変化させるかを比較する。

| パターン | フィルタ数の推移 | 仮説 |
|---------|----------------|------|
| 増やす (定石) | 16 → 32 → 64 | 深い層ほど複雑な特徴 → 多くのフィルタが必要 |
| 一定 | 32 → 32 → 32 | フィルタ数は変えなくてもいい？ |
| 減らす (逆) | 64 → 32 → 16 | 定石の逆。精度が落ちるはず |

In [ ]:
class CNN_3Layer_Custom(nn.Module):
    def __init__(self, f1, f2, f3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, f1, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 28→14
            nn.Conv2d(f1, f2, 3, padding=1), nn.ReLU(),
            nn.Conv2d(f2, f3, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 14→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(f3 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


scaling_models = {
    "増やす (16→32→64)": CNN_3Layer_Custom(16, 32, 64),
    "一定 (32→32→32)": CNN_3Layer_Custom(32, 32, 32),
    "減らす (64→32→16)": CNN_3Layer_Custom(64, 32, 16),
}

scaling_results = run_experiment(scaling_models, "実験4: フィルタ数の増やし方")

In [ ]:
plot_comparison(scaling_results, "実験4: フィルタ数の増やし方")

---
## 全実験サマリー

In [ ]:
print("=" * 60)
print("全実験サマリー")
print("=" * 60)

all_experiments = [
    ("実験1: フィルタ数", filter_results),
    ("実験2: フィルタサイズ", kernel_results),
    ("実験3: 層の深さ", depth_results),
    ("実験4: フィルタ数の増やし方", scaling_results),
]

for exp_name, results in all_experiments:
    print(f"\n{exp_name}")
    print("-" * 50)
    for name, data in results.items():
        print(f"  {name:<25s}  正解率: {data['accuracy']:.2f}%  パラメータ: {data['params']:>10,}")

    best = max(results, key=lambda k: results[k]["accuracy"])
    print(f"  → ベスト: {best}")